# S6E8 — Stacking massif + features polynomiales (reproduction technique gagnante)

Reproduit l'approche LB 0.97059 / CV 0.96947 (mhamza0810) :
1. Charge **15 meta-features** (OOF + test preds de notebooks publics) depuis le dataset `adnaneel/s6e8-meta-features`
2. Ajoute des **interactions polynomiales** (degré 2) des meta-features
3. Entraîne un **XGBoost lent** (lr=0.01, 20000 arbres) sur **GPU**

Version 2 : 15 modèles (GBDT + NN + AutoML + lookup transformer).

In [ ]:
import os, warnings, gc
import numpy as np
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
import torch
warnings.filterwarnings('ignore')

PATH = '/kaggle/input/competitions/playground-series-s6e8'
N_FOLDS = 5
SEED = 42
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

train = pd.read_csv(f'{PATH}/train.csv')
test = pd.read_csv(f'{PATH}/test.csv')
sub = pd.read_csv(f'{PATH}/sample_submission.csv')
print('train', train.shape, 'test', test.shape)

In [ ]:
# Charger les meta-features TOP (sans resnet, flaml_lgb, lgb_don - les moins prédictives)
META_TRAIN = pd.read_csv('/kaggle/input/s6e8-meta-features/meta_train_top.csv')
META_TEST = pd.read_csv('/kaggle/input/s6e8-meta-features/meta_test_top.csv')
print('meta train:', META_TRAIN.shape, '| meta test:', META_TEST.shape)
print('features:', [c for c in META_TRAIN.columns if c != 'id'])

META = {}
for c in META_TRAIN.columns:
    if c != 'id':
        META[c] = (META_TRAIN[c].values, META_TEST[c].values)
print('\nmeta-features chargées:', list(META.keys()))

In [ ]:
# Construire les meta-features + interactions polynomiales
train_v2 = train.copy()
test_v2 = test.copy()
for name, (oof, tp) in META.items():
    train_v2[f'preds_{name}'] = oof
    test_v2[f'preds_{name}'] = tp

meta_cols = [c for c in train_v2.columns if c.startswith('preds_')]
print('meta features:', len(meta_cols))

poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
train_poly = poly.fit_transform(train_v2[meta_cols])
test_poly = poly.transform(test_v2[meta_cols])
poly_cols = poly.get_feature_names_out(meta_cols)
train_poly = pd.DataFrame(train_poly, columns=poly_cols, index=train_v2.index).drop(columns=meta_cols)
test_poly = pd.DataFrame(test_poly, columns=poly_cols, index=test_v2.index).drop(columns=meta_cols)
print('features polynomiales:', train_poly.shape[1])

train_v3 = pd.concat([train_v2, train_poly], axis=1)
test_v3 = pd.concat([test_v2, test_poly], axis=1)

In [ ]:
FEATURES = [c for c in train_v3.columns if c not in ['id', 'addicted_label']]
X = train_v3[FEATURES]
y = train_v3['addicted_label']
X_test = test_v3[FEATURES]
cc = X.select_dtypes(include='object').columns
for c in cc:
    X[c] = X[c].astype('category')
    X_test[c] = X_test[c].astype('category')
print('features finales:', X.shape)

In [ ]:
params = dict(
    objective='binary:logistic', eval_metric='auc', tree_method='hist',
    device='cuda', enable_categorical=True,
    learning_rate=0.01, max_depth=8, min_child_weight=10,
    subsample=0.8, colsample_bytree=0.5, reg_alpha=0.1, reg_lambda=1.0,
    n_estimators=20000, random_state=SEED, max_bin=2000,
)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
fold_aucs = []

for fold, (tr, va) in enumerate(skf.split(X, y)):
    print(f'\nFold {fold+1}', flush=True)
    m = XGBClassifier(**params, missing=np.nan, early_stopping_rounds=200)
    m.fit(X.iloc[tr], y.iloc[tr], eval_set=[(X.iloc[va], y.iloc[va])], verbose=200)
    oof[va] = m.predict_proba(X.iloc[va])[:, 1]
    test_preds += m.predict_proba(X_test)[:, 1] / N_FOLDS
    fa = roc_auc_score(y.iloc[va], oof[va])
    fold_aucs.append(fa)
    print(f'Fold {fold+1} AUC: {fa:.5f}', flush=True)
    gc.collect()

print(f'\nMean CV AUC: {np.mean(fold_aucs):.5f} | OOB: {roc_auc_score(y, oof):.5f}')

In [ ]:
sub['addicted_label'] = test_preds
sub.to_csv('submission.csv', index=False)
np.save('oof_stack.npy', oof)
np.save('test_stack.npy', test_preds)
print('submission prête')
sub.head()